# Credit Risk Guru — the `ear` package, reasoning at runtime

This notebook uses [`ear`](../ear/__init__.py), where every class, field
and message is natively English, and where every judgment-laden stage — `Policy` compliance, `Discoverer` relevance,
`Reasoner`'s decision, `Explainer`'s prose — reasons in natural language
against a live Claude model via DSPy, falling back to a small
dependency-free default only when no `ModelBinding` is active.

Two things distinguish this notebook:

1. **`Policy` is written in plain English**, e.g. *"The applicant's
   debt-to-income ratio must not exceed 0.45"* — judged by the LLM, not a
   hardcoded boolean expression. A `fallback_expression` exists only as a
   deterministic safety net for when no model is configured, never as the
   primary path.
2. **Applicant scenarios are not hardcoded.** There is no literal Python
   dict of "Asha Rao, score 760, DTI 0.22" anywhere below. A DSPy signature
   (`GenerateApplicantScenario`) invents each applicant from a one-line
   steer at call time, reasoning fresh every run; without a model
   configured, a procedural (not literal) fallback varies applicants
   instead.

| `ear` class | Credit risk role |
| --- | --- |
| `Skill` | One scoring capability (score band, DTI band, risk grade) |
| `Persona` | The credit analyst persona carrying out the judgement |
| `Workflow` | The underwriting workflow ordering personas |
| `Process` | "Personal Loan Underwriting", with a `description` the `Discoverer` reasons over |
| `Policy` | Hard underwriting / fair-lending floors, written in plain English |
| `Runtime` | Runs the full Governor → ... → Adapter pipeline every cycle |
| `ModelBinding` | The live Claude binding every LLM-judged stage reasons against |
| `Evidence` | The audit trail behind one decision |
| `Memory` / `Experience` / `Adaptation` | What happened / the pattern / the distilled lesson |


## 0. Imports

Read the Anthropic API key only from the environment — never hardcode a
key in a notebook cell or commit one to a file. If `ANTHROPIC_API_KEY`
isn't set, every LLM-judged stage below falls back to its dependency-free
default and this notebook still runs end to end.

In [1]:
import os

from ear import (
    Intent,
    ModelBinding,
    Persona,
    Policy,
    Process,
    Runtime,
    Skill,
    Workflow,
)

print("ear package ready --", "ANTHROPIC_API_KEY is set" if os.environ.get("ANTHROPIC_API_KEY") else "no ANTHROPIC_API_KEY set (deterministic fallbacks will run)")

ear package ready -- no ANTHROPIC_API_KEY set (deterministic fallbacks will run)


## 1. `Skill` — feature-engineering capabilities

These are deterministic by design — banding a FICO score or a DTI ratio
into a tier is arithmetic, not judgment, so there is nothing for an LLM to
reason about here. The judgment happens later, in `Policy` and `Reasoner`.

In [2]:
def credit_score_tier(score: int) -> str:
    if score >= 720:
        return "prime"
    if score >= 660:
        return "near-prime"
    if score >= 620:
        return "subprime"
    return "deep-subprime"


def dti_tier(debt_to_income_ratio: float) -> str:
    if debt_to_income_ratio <= 0.30:
        return "low"
    if debt_to_income_ratio <= 0.45:
        return "moderate"
    return "high"


_RISK_GRID = {
    ("prime", "low"): "A", ("prime", "moderate"): "A", ("prime", "high"): "B",
    ("near-prime", "low"): "B", ("near-prime", "moderate"): "B", ("near-prime", "high"): "C",
    ("subprime", "low"): "C", ("subprime", "moderate"): "C", ("subprime", "high"): "D",
    ("deep-subprime", "low"): "D", ("deep-subprime", "moderate"): "D", ("deep-subprime", "high"): "E",
}


def risk_grade(score_tier: str, dti_band: str) -> str:
    return _RISK_GRID.get((score_tier, dti_band), "E")


score_skill = Skill(name="credit_score_tier", description="Bands a FICO score", handler=credit_score_tier)
dti_skill = Skill(name="dti_tier", description="Bands a debt-to-income ratio", handler=dti_tier)
grade_skill = Skill(name="risk_grade", description="Combines score + DTI tiers into a risk grade", handler=risk_grade)

print([s.name for s in (score_skill, dti_skill, grade_skill)])

['credit_score_tier', 'dti_tier', 'risk_grade']


## 2. `Persona`, `Workflow`, `Process`

`Process.description` is new relative to the original package — it gives
the `Discoverer` a natural-language description to reason over, instead of
only the bare process name.

In [3]:
credit_risk_guru = Persona(
    name="Credit Risk Guru",
    instructions=(
        "Underwrite personal loans conservatively. Band every applicant's "
        "credit score and debt-to-income ratio, then assign a risk grade "
        "before any approval reasoning happens."
    ),
)
credit_risk_guru.add_skill(score_skill).add_skill(dti_skill).add_skill(grade_skill)

underwriting_workflow = Workflow(name="Underwriting Workflow")
underwriting_workflow.add_persona(credit_risk_guru)

loan_underwriting = Process(
    name="Personal Loan Underwriting",
    description=(
        "Reviews a personal loan applicant's credit score, debt-to-income "
        "ratio, income and existing defaults to decide whether to approve "
        "the requested loan amount."
    ),
)
loan_underwriting.add_workflow(underwriting_workflow)

print(loan_underwriting.name, "->", loan_underwriting.description)

Personal Loan Underwriting -> Reviews a personal loan applicant's credit score, debt-to-income ratio, income and existing defaults to decide whether to approve the requested loan amount.


## 3. `Policy` — underwriting floors, written in plain English

Each `Policy.statement` is what the LLM actually judges against the
intent's context once a `ModelBinding` is active — that's the primary,
natural-language path. `fallback_expression` is a safe-eval expression
(never `eval`/`exec`) used only when no model is configured, so governance
never silently passes through just because a provider wasn't wired up.

In [4]:
policies = [
    Policy(
        name="Minimum Credit Score",
        statement="The applicant's FICO credit score must be at least 620.",
        fallback_expression="credit_score >= 620",
    ),
    Policy(
        name="Debt-to-Income Ceiling",
        statement="The applicant's debt-to-income ratio must not exceed 0.45.",
        fallback_expression="debt_to_income_ratio <= 0.45",
    ),
    Policy(
        name="No Active Defaults",
        statement="The applicant must have no existing loan defaults on record.",
        fallback_expression="existing_defaults == 0",
    ),
    Policy(
        name="Loan Amount Cap",
        statement="The requested personal loan amount must not exceed $75,000.",
        fallback_expression="loan_amount <= 75000",
    ),
]

for policy in policies:
    print(f"- {policy.name}: {policy.statement}")

- Minimum Credit Score: The applicant's FICO credit score must be at least 620.
- Debt-to-Income Ceiling: The applicant's debt-to-income ratio must not exceed 0.45.
- No Active Defaults: The applicant must have no existing loan defaults on record.
- Loan Amount Cap: The requested personal loan amount must not exceed $75,000.


## 4. `ModelBinding` — give the Guru a real mind, from the environment only

Reads `ANTHROPIC_API_KEY` exclusively via `os.environ` at call time —
`ModelBinding.resolve_api_key()` never has a key passed to it as a
literal. With no key set, `model_binding` stays `None` and every
LLM-judged stage below uses its dependency-free default instead.

In [5]:
model_binding = None
if os.environ.get("ANTHROPIC_API_KEY"):
    model_binding = ModelBinding(provider="anthropic", model=os.environ.get("ANTHROPIC_TEST_MODEL", "claude-haiku-4-5"))
    model_binding.activate()
    print(f"ModelBinding attached -- reasoning against {model_binding.model_id}.")
else:
    print(
        "No ANTHROPIC_API_KEY in the environment -- Policy, Discoverer, Reasoner and "
        "Explainer all fall back to their dependency-free defaults, and applicant "
        "scenarios below come from the procedural (not literal) fallback generator."
    )

No ANTHROPIC_API_KEY in the environment -- Policy, Discoverer, Reasoner and Explainer all fall back to their dependency-free defaults, and applicant scenarios below come from the procedural (not literal) fallback generator.


## 5. `Runtime` — assemble the pipeline

`Runtime` wires in all nineteen pipeline stages itself; we only register
the process, the policies, and the `ModelBinding`.

In [6]:
runtime = Runtime(name="Credit-Risk-Guru-Runtime", model_binding=model_binding)
runtime.add_process(loan_underwriting)
for policy in policies:
    runtime.add_policy(policy)

print(runtime.name)
print("Processes:", [p.name for p in runtime.processes])
print("Policies: ", [p.name for p in runtime.policies])

Credit-Risk-Guru-Runtime
Processes: ['Personal Loan Underwriting']
Policies:  ['Minimum Credit Score', 'Debt-to-Income Ceiling', 'No Active Defaults', 'Loan Amount Cap']


## 6. Generating applicant scenarios — prompt and runtime reasoning, not a hardcoded list

`GenerateApplicantScenario` is a DSPy signature that invents one applicant
from a one-line steer (`portfolio_brief`), reasoning fresh on every call —
there is no literal applicant dict anywhere in this notebook. When no
`ModelBinding` is active, `_fallback_scenario` varies an applicant
procedurally from the same brief (using `random`, not a fixed literal), so
the notebook still runs end to end without an LLM, exactly mirroring the
rest of `ear`'s "LLM if configured, else dependency-free default"
philosophy.

In [7]:
import random

import dspy


class GenerateApplicantScenario(dspy.Signature):
    """Invent one realistic personal-loan applicant scenario for an
    underwriting demo. Vary the credit profile and narrative each time so
    repeated calls produce a diverse portfolio, not a repeated template."""

    portfolio_brief: str = dspy.InputField(
        desc="A one-line steer on what kind of applicant to invent next"
    )
    applicant_name: str = dspy.OutputField(desc="A plausible full name")
    credit_score: int = dspy.OutputField(desc="A FICO score between 300 and 850")
    debt_to_income_ratio: float = dspy.OutputField(desc="A debt-to-income ratio between 0.0 and 1.0")
    annual_income: float = dspy.OutputField(desc="Annual income in US dollars")
    loan_amount: float = dspy.OutputField(desc="Requested personal loan amount in US dollars")
    existing_defaults: int = dspy.OutputField(desc="Number of existing loan defaults on record")
    narrative: str = dspy.OutputField(desc="One sentence on why this applicant is applying")


_PORTFOLIO_BRIEFS = [
    "a clean prime-tier applicant who should comfortably clear every policy",
    "a borderline applicant hovering right at the debt-to-income ceiling",
    "a subprime applicant who has missed payments before but carries no active defaults",
    "an applicant whose debt-to-income ratio breaches the ceiling and must be blocked before reasoning starts",
    "a high-income applicant requesting a large loan near the policy cap",
]


def _fallback_scenario(brief: str, rng: random.Random) -> dict:
    breach_dti = "breach" in brief or "blocked" in brief
    return {
        "applicant_name": f"Applicant {rng.randint(1000, 9999)}",
        "credit_score": rng.randint(640, 760) if breach_dti else rng.randint(600, 820),
        "debt_to_income_ratio": round(rng.uniform(0.50, 0.65), 2) if breach_dti else round(rng.uniform(0.15, 0.44), 2),
        "annual_income": float(rng.randint(40_000, 150_000)),
        "loan_amount": float(rng.randint(5_000, 70_000)),
        "existing_defaults": 0,
        "narrative": f"Generated by the dependency-free fallback from the brief: {brief}",
    }


def generate_applicant_scenario(brief: str, model_binding=None, rng: "random.Random | None" = None) -> dict:
    """Invent one applicant scenario -- via the LLM when `model_binding`
    is active, else a procedurally varied fallback. Never reads from a
    hardcoded applicant list."""
    rng = rng or random.Random()
    if model_binding is not None and model_binding.lm is not None:
        program = dspy.Predict(GenerateApplicantScenario)
        with dspy.context(lm=model_binding.lm):
            prediction = program(portfolio_brief=brief)
        return {
            "applicant_name": prediction.applicant_name,
            "credit_score": int(prediction.credit_score),
            "debt_to_income_ratio": float(prediction.debt_to_income_ratio),
            "annual_income": float(prediction.annual_income),
            "loan_amount": float(prediction.loan_amount),
            "existing_defaults": int(prediction.existing_defaults),
            "narrative": prediction.narrative,
        }
    return _fallback_scenario(brief, rng)


rng = random.Random(2026)
scenarios = [generate_applicant_scenario(brief, model_binding, rng) for brief in _PORTFOLIO_BRIEFS]
for scenario in scenarios:
    print(f"- {scenario['applicant_name']}: score={scenario['credit_score']}, DTI={scenario['debt_to_income_ratio']}, loan=${scenario['loan_amount']:,.0f} -- {scenario['narrative']}")

- Applicant 2951: score=681, DTI=0.3, loan=$47,414 -- Generated by the dependency-free fallback from the brief: a clean prime-tier applicant who should comfortably clear every policy
- Applicant 2681: score=657, DTI=0.41, loan=$41,474 -- Generated by the dependency-free fallback from the brief: a borderline applicant hovering right at the debt-to-income ceiling
- Applicant 7891: score=800, DTI=0.32, loan=$55,893 -- Generated by the dependency-free fallback from the brief: a subprime applicant who has missed payments before but carries no active defaults
- Applicant 9042: score=736, DTI=0.62, loan=$20,724 -- Generated by the dependency-free fallback from the brief: an applicant whose debt-to-income ratio breaches the ceiling and must be blocked before reasoning starts
- Applicant 1041: score=757, DTI=0.17, loan=$58,527 -- Generated by the dependency-free fallback from the brief: a high-income applicant requesting a large loan near the policy cap


## 7. Turning a generated scenario into an `Intent`

Feature computation (score banding, DTI banding, risk grading) still
happens through the Guru's own `Skill`s before the runtime reasons — the
policy gates and the explanation both see the same banded features a human
underwriter would, regardless of whether the raw numbers came from the LLM
or the fallback generator.

In [8]:
def scenario_to_intent(scenario: dict) -> Intent:
    score_tier = credit_risk_guru.get_skill("credit_score_tier").invoke(scenario["credit_score"])
    dti_band = credit_risk_guru.get_skill("dti_tier").invoke(scenario["debt_to_income_ratio"])
    grade = credit_risk_guru.get_skill("risk_grade").invoke(score_tier, dti_band)

    context = {
        "credit_score": scenario["credit_score"],
        "debt_to_income_ratio": scenario["debt_to_income_ratio"],
        "annual_income": scenario["annual_income"],
        "loan_amount": scenario["loan_amount"],
        "existing_defaults": scenario["existing_defaults"],
        "score_tier": score_tier,
        "dti_band": dti_band,
        "risk_grade": grade,
    }
    text = (
        f"Underwrite a ${scenario['loan_amount']:,.0f} personal loan for "
        f"{scenario['applicant_name']} (risk grade {grade}). {scenario['narrative']}"
    )
    return Intent(text=text, context=context)


def show_decision(name: str, entry) -> None:
    print(f"Applicant: {name}")
    print(f"  Decision : {entry.decision}")
    print(f"  Basis    : {entry.evidence.basis}")
    print(f"  Plan     : {entry.evidence.sources['plan']}")
    print(f"  Policies checked: {entry.evidence.sources['policies_checked']}")
    print(f"  Explanation: {entry.evidence.sources['explanation']}")

## 8. Running the generated portfolio through `Runtime.reason()`

`Governor` checks every `Policy` first — before `Discoverer` even
discovers a process — so an applicant whose debt-to-income ratio breaches
the ceiling never reaches reasoning at all; `Runtime.reason()` raises
`PermissionError` immediately and no `Memory` entry is written.

In [9]:
for scenario in scenarios:
    intent = scenario_to_intent(scenario)
    try:
        runtime.reason(intent)
        show_decision(scenario["applicant_name"], runtime.memory.working[-1])
    except PermissionError as exc:
        print(f"Applicant: {scenario['applicant_name']}")
        print(f"  Rejected before reasoning even started: {exc}")
    print()

print(f"Total successful cycles observed by Experience: {runtime.experience.observations}")

Applicant: Applicant 2951
  Decision : [Credit-Risk-Guru-Runtime] resolved intent 'Underwrite a $47,414 personal loan for Applicant 2951 (risk grade B). Generated by the dependency-free fallback from the brief: a clean prime-tier applicant who should comfortably clear every policy' across processes: Personal Loan Underwriting
  Basis    : Resolved via the Reasoner's dependency-free default
  Plan     : ['Underwriting Workflow']
  Policies checked: ['Minimum Credit Score', 'Debt-to-Income Ceiling', 'No Active Defaults', 'Loan Amount Cap']
  Explanation: Resolved via the Reasoner's dependency-free default -> [Credit-Risk-Guru-Runtime] resolved intent 'Underwrite a $47,414 personal loan for Applicant 2951 (risk grade B). Generated by the dependency-free fallback from the brief: a clean prime-tier applicant who should comfortably clear every policy' across processes: Personal Loan Underwriting

Applicant: Applicant 2681
  Decision : [Credit-Risk-Guru-Runtime] resolved intent 'Underwrite a 

## 9. Inspecting `Memory`, `Experience` and `Adaptation`

`Adapter`'s default `adapt_every=5` means a fifth successful cycle (if
reached) triggers `AdaptationBank` to distill one new lesson from
`Experience`'s aggregated pattern — not on every single call.

In [10]:
print("=== Memory (what happened) ===")
print(runtime.memory.context_window())

print("\n=== Experience (the pattern across repeated cycles) ===")
print(runtime.experience.summary())

print("\n=== Adaptation (the distilled lesson) ===")
if runtime.adaptations.impressions:
    for impression in runtime.adaptations.impressions:
        print(f"- {impression.name}: {impression.insight}")
else:
    print("No adaptation distilled yet.")

=== Memory (what happened) ===
Recent history:
- (2026-06-25 08:14) 'Underwrite a $47,414 personal loan for Applicant 2951 (risk grade B). Generated by the dependency-free fallback from the brief: a clean prime-tier applicant who should comfortably clear every policy' -> [Credit-Risk-Guru-Runtime] resolved intent 'Underwrite a $47,414 personal loan for Applicant 2951 (risk grade B). Generated by the dependency-free fallback from the brief: a clean prime-tier applicant who should comfortably clear every policy' across processes: Personal Loan Underwriting [Resolved via the Reasoner's dependency-free default]
- (2026-06-25 08:14) 'Underwrite a $41,474 personal loan for Applicant 2681 (risk grade C). Generated by the dependency-free fallback from the brief: a borderline applicant hovering right at the debt-to-income ceiling' -> [Credit-Risk-Guru-Runtime] resolved intent 'Underwrite a $41,474 personal loan for Applicant 2681 (risk grade C). Generated by the dependency-free fallback from th

## 10. The full `Evidence` audit trail behind one decision

`Evidence` records which reasoning path was used, which policies were
checked, the composed plan, what `Recaller` recalled from memory, and
`Explainer`'s explanation — separately from the decision itself.

In [11]:
import json

if runtime.memory.working:
    last_entry = runtime.memory.working[-1]
    print(json.dumps(
        {**last_entry.evidence.sources, "decision": last_entry.decision, "basis": last_entry.evidence.basis},
        indent=2, default=str,
    ))
else:
    print("No successful cycle was recorded -- every generated applicant breached a Policy.")

{
  "policies_checked": [
    "Minimum Credit Score",
    "Debt-to-Income Ceiling",
    "No Active Defaults",
    "Loan Amount Cap"
  ],
  "context": {
    "credit_score": 757,
    "debt_to_income_ratio": 0.17,
    "annual_income": 77649.0,
    "loan_amount": 58527.0,
    "existing_defaults": 0,
    "score_tier": "prime",
    "dti_band": "low",
    "risk_grade": "A"
  },
  "plan": [
    "Underwriting Workflow"
  ],
  "recalled_memory": "Recent history:\n- (2026-06-25 08:14) 'Underwrite a $47,414 personal loan for Applicant 2951 (risk grade B). Generated by the dependency-free fallback from the brief: a clean prime-tier applicant who should comfortably clear every policy' -> [Credit-Risk-Guru-Runtime] resolved intent 'Underwrite a $47,414 personal loan for Applicant 2951 (risk grade B). Generated by the dependency-free fallback from the brief: a clean prime-tier applicant who should comfortably clear every policy' across processes: Personal Loan Underwriting [Resolved via the Reasoner's

## Why this notebook reasons at runtime instead of hardcoding

- **No hardcoded applicant data.** `GenerateApplicantScenario` invents
  every applicant from a one-line steer at call time; the fallback path
  varies applicants procedurally instead of reading a fixed literal dict.
- **`Policy` is judged in natural language.** `Policy.statement` is what
  the LLM reasons over when a `ModelBinding` is active; the safe-eval
  `fallback_expression` only backstops the rare case where no model is
  configured, so governance never silently no-ops.
- **DSPy and GEPA are used sparingly.** Only the genuinely judgment-laden
  stages (`Policy`, `Discoverer`, `Reasoner`, `Explainer`, and this
  notebook's scenario generator) carry a DSPy signature; `Selector`,
  `Composer` and `Scheduler` stay plain Python because there is no
  judgment call to make in deduplicating, flattening or copying a list.
  GEPA itself is reserved for `Reasoner.optimize_with_gepa()` alone — not
  invoked in this notebook, since optimizing a reasoning program is a
  development-time exercise, not a per-cycle one.
- **Every class is native English**: `ear.Runtime`, `ear.Policy`,
  `ear.ModelBinding`, and so on are real classes defined in `ear/`.
